In [1]:
# Importando as bibliotecas.
import gc
import pytz
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, when
from pyspark.sql.types import StringType, FloatType
from googletrans import Translator
from nltk.sentiment import SentimentIntensityAnalyzer
from datetime import datetime

In [2]:
# Configurando logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [3]:
# Configurações de variáveis para o BigQuery
bucket_dataproc = 'project-b2d9e7e0-e964-49fe-935-dataproc'
nome_tabela_analizada_bq = 'tb_emails_feedback_analizados'
projeto_bigquery = 'project-b2d9e7e0-e964-49fe-935'
dataset_bigquery = 'insight_data'
tabela_origem = 'tb_emails_feedback'

In [4]:
# Inicializar a Spark Session
logging.info("Inicializando a Spark Session com as configurações definidas...")
spark = SparkSession.builder \
    .appName("Email Feedback Analysis with NLTK") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.instances", "3") \
    .config('spark.jars.packages', 'com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.28.0') \
    .getOrCreate()
spark.conf.set("temporaryGcsBucket", bucket_dataproc)
spark.conf.set("viewsEnabled", "true")
spark.conf.set("materializationDataset", dataset_bigquery)
spark.conf.set("spark.sql.debug.maxToStringFields", 1000)

2026-05-09 17:32:59,610 - INFO - Inicializando a Spark Session com as configurações definidas...
26/05/09 17:33:01 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
# Instâncias para tradução e análise de sentimentos
logging.info("Inicializando a instância do analisador de sentimentos...")
sia = SentimentIntensityAnalyzer()
logging.info("Analisador de sentimentos inicializado com sucesso.")

2026-05-09 17:33:49,865 - INFO - Inicializando a instância do analisador de sentimentos...
2026-05-09 17:33:49,884 - INFO - Analisador de sentimentos inicializado com sucesso.


In [ ]:
# Função para traduzir o texto
def translate_text(text):
    translator = Translator()
    try:
        lang = translator.detect(text).lang
        if lang != 'en':
            return translator.translate(text, src=lang, dest='en').text
        else:
            return text
    except Exception:
        return text  # Retorna o texto original em caso de falha na tradução

In [7]:
# Função para análise de sentimentos
def analyze_sentiment(text):
    sentiment = sia.polarity_scores(text)['compound']
    return sentiment

In [8]:
# Criar UDFs para tradução e análise de sentimento
logging.info("Criando UDFs para tradução e análise de sentimentos...")
translate_udf = udf(translate_text, StringType())
sentiment_udf = udf(analyze_sentiment, FloatType())
logging.info("UDFs criadas com sucesso.")

2026-05-09 17:36:38,082 - INFO - Criando UDFs para tradução e análise de sentimentos...
2026-05-09 17:36:38,088 - INFO - UDFs criadas com sucesso.


In [11]:
# Leitura da tabela no BigQuery e conversão em um DF Spark
full_table_name_emails = f'{projeto_bigquery}.{dataset_bigquery}.{tabela_origem}'
df_email = spark.read.format('bigquery').option('table', full_table_name_emails).load()
logging.info(f"Tabela {tabela_origem} lida com sucesso. Iniciando a análise de sentimentos...")

2026-05-09 17:45:15,566 - INFO - Tabela tb_emails_feedback lida com sucesso. Iniciando a análise de sentimentos...


In [12]:
df_email.show(5)

+--------------------+--------------------+--------------------+--------------------+--------------------+
|           remetente|        destinatario|             assunto|                data|               corpo|
+--------------------+--------------------+--------------------+--------------------+--------------------+
|gregory88@example...|tinaadams@example...|Feedback sobre es...|Sun, 08 Sep 2024 ...|A cidade estava s...|
| kayla33@example.net|lewisphyllis@exam...|Feedback sobre es...|Sun, 29 Sep 2024 ...|A cidade estava s...|
|pattersondaniel@e...|   cclay@example.com|Feedback sobre es...|Sat, 28 Sep 2024 ...|A cidade estava s...|
|josephjames@examp...|carpenterangela@e...|Feedback sobre es...|Mon, 16 Sep 2024 ...|A cidade estava s...|
|margaret17@exampl...|  alex54@example.net|Feedback sobre es...|Wed, 25 Sep 2024 ...|A cidade foi dece...|
+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows



In [13]:
# Aplicar UDFs para traduzir e calcular pontuação de sentimento
logging.info("Aplicando tradução e cálculo de sentimento nos textos...")
df_analizado = df_email.withColumn('translated_body', translate_udf(col('corpo')))
df_analizado = df_analizado.withColumn('sentiment_score', sentiment_udf(col('translated_body')))
df_analizado = df_analizado.drop("translated_body")
logging.info("Tradução e cálculo de sentimento concluídos.")

2026-05-09 17:46:17,503 - INFO - Aplicando tradução e cálculo de sentimento nos textos...
2026-05-09 17:46:18,134 - INFO - Tradução e cálculo de sentimento concluídos.


In [14]:
# Classificar os sentimentos como 'Positive', 'Negative', ou 'Neutral'
logging.info("Classificando os sentimentos como Positive, Negative ou Neutral...")
df_analizado = df_analizado.withColumn('sentiment',
                   when(col('sentiment_score') > 0, 'Positive')
                   .when(col('sentiment_score') < 0, 'Negative')
                   .otherwise('Neutral'))
logging.info("Classificação dos sentimentos concluída.")

2026-05-09 17:47:32,810 - INFO - Classificando os sentimentos como Positive, Negative ou Neutral...
2026-05-09 17:47:32,922 - INFO - Classificação dos sentimentos concluída.


In [17]:
# Marcar o início da execução
fuso_horario_sp = pytz.timezone('America/Sao_Paulo')
inicio = datetime.now(fuso_horario_sp)
logging.info(f"Início da carga: {inicio}")

2026-05-09 17:54:44,563 - INFO - Início da carga: 2026-05-09 14:54:44.562999-03:00


In [18]:
# Carregar o resultado da análise no BigQuery
logging.info("Carregando os resultados da análise de sentimentos no BigQuery...")
df_analizado.write.format('bigquery') \
.mode("overwrite") \
.option('table', nome_tabela_analizada_bq) \
.option('parentProject', projeto_bigquery) \
.option('dataset', dataset_bigquery) \
.option('temporaryGcsBucket', bucket_dataproc) \
.save()
logging.info('Análise de sentimentos dos emails carregado com sucesso no BigQuery')

2026-05-09 17:54:44,650 - INFO - Carregando os resultados da análise de sentimentos no BigQuery...
2026-05-09 18:00:25,650 - INFO - Análise de sentimentos dos emails carregado com sucesso no BigQuery


In [19]:
# Marcar o fim da execução
fim = datetime.now(fuso_horario_sp)
logging.info(f"Fim da carga: {fim}")


# Calcular a duração da execução
duracao = fim - inicio
logging.info(f"Duração total: {duracao}")


# Excluindo o DF Spark e liberando a memória do cluster
df_email.unpersist()
df_analizado.unpersist()
del df_email,df_analizado
gc.collect()
spark.stop()
logging.info('Memória do cluster liberada e Spark finalizado.')

2026-05-09 18:00:25,662 - INFO - Fim da carga: 2026-05-09 15:00:25.661922-03:00
2026-05-09 18:00:25,665 - INFO - Duração total: 0:05:41.098923
2026-05-09 18:00:26,881 - INFO - Memória do cluster liberada e Spark finalizado.
